# Simulación del Protocolo BB84

Implementación completa del protocolo de distribución cuántica de claves BB84.

**Objetivos:**
- Simular el protocolo BB84 con y sin espía
- Verificar la relación entre la tasa de intercepción de Eva y el QBER medido
- Estimar la tasa de clave segura con y sin amplificación de privacidad
- Visualizar el protocolo con un diagrama de secuencia

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple

rng = np.random.default_rng(seed=42)
print('Librerías cargadas')

## 1. Implementación del Protocolo BB84

In [ ]:
def bb84_protocol(
    n_bits: int,
    eve_intercept_prob: float = 0.0,
    verbose: bool = False,
) -> Tuple[np.ndarray, np.ndarray, float, float]:
    """
    Simula el protocolo BB84.
    
    Args:
        n_bits: Número de bits transmitidos
        eve_intercept_prob: Probabilidad de que Eva intercepte cada fotón
        verbose: Mostrar detalles del protocolo
    
    Returns:
        (alice_key, bob_key, qber, key_rate)
    """
    # Paso 1: Alice genera bits y bases aleatorias
    alice_bits  = rng.integers(0, 2, n_bits)           # bits a transmitir
    alice_bases = rng.integers(0, 2, n_bits)           # 0=+ (computacional), 1=× (Hadamard)
    
    # Paso 2: Bob elige bases de medición aleatorias
    bob_bases = rng.integers(0, 2, n_bits)
    
    # Paso 3: Eva intercepta con probabilidad eve_intercept_prob
    # Eva mide en una base aleatoria y reenvía su resultado
    received_by_bob = alice_bits.copy()
    eva_mask  = rng.random(n_bits) < eve_intercept_prob
    eve_bases = rng.integers(0, 2, n_bits)
    
    for i in range(n_bits):
        if eva_mask[i]:
            # Eva mide: si su base coincide con Alice, obtiene el bit correcto
            # si no, obtiene un bit aleatorio y perturba el fotón
            if eve_bases[i] != alice_bases[i]:
                # Eva perturbó el fotón → Bob obtiene un bit aleatorio
                received_by_bob[i] = rng.integers(0, 2)
    
    # Paso 4: Bob mide en su base
    # Si la base de Bob coincide con la de Alice (post-Eva), obtiene el bit correcto
    # Si no, obtiene un bit aleatorio
    bob_bits = np.where(
        bob_bases == alice_bases,
        received_by_bob,           # bases coinciden: bit correcto (o perturbado por Eva)
        rng.integers(0, 2, n_bits) # bases no coinciden: aleatorio
    )
    
    # Paso 5: Sifting — conservar solo los bits donde las bases coinciden
    matching = alice_bases == bob_bases
    alice_key = alice_bits[matching]
    bob_key   = bob_bits[matching]
    key_rate  = np.sum(matching) / n_bits
    
    # Paso 6: Calcular QBER (en un subconjunto de la clave)
    qber = float(np.mean(alice_key != bob_key))
    
    if verbose:
        print(f'Bits transmitidos: {n_bits}')
        print(f'Bits en clave cruda: {len(alice_key)} ({key_rate:.1%})')
        print(f'QBER: {qber:.4f} ({qber*100:.2f}%)')
        print(f'Bits interceptados por Eva: {np.sum(eva_mask)}')
        n_eva_errors = np.sum(eva_mask & (alice_bases != eve_bases))
        print(f'Perturbaciones de Eva: {n_eva_errors}')
    
    return alice_key, bob_key, qber, key_rate

# Test sin espía
print('=== Sin espía ===')
ak, bk, qber, rate = bb84_protocol(10000, eve_intercept_prob=0.0, verbose=True)
print(f'¿Claves idénticas? {np.array_equal(ak, bk)}')

print('\n=== Con espía (100%) ===')
ak, bk, qber, rate = bb84_protocol(10000, eve_intercept_prob=1.0, verbose=True)
print(f'QBER esperado teóricamente: 25% (1/4 de los bits son perturbados por Eva)')

## 2. QBER vs Tasa de Intercepción de Eva

In [ ]:
n_bits = 50000
eve_probs = np.linspace(0, 1, 20)
qbers  = []
rates  = []

for p_eve in eve_probs:
    _, _, qber, rate = bb84_protocol(n_bits, eve_intercept_prob=p_eve)
    qbers.append(qber)
    rates.append(rate)

# Teoría: QBER = p_eve / 4 (ya que Eva solo perturba cuando usa base incorrecta = 50%)
# y Bob elige base correcta 50% → QBER = p_eve × 0.5 × 0.5 = p_eve/4
qber_theory = [p/4 for p in eve_probs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(eve_probs * 100, np.array(qbers) * 100, 'o-', color='#e74c3c', label='Simulación')
ax1.plot(eve_probs * 100, np.array(qber_theory) * 100, '--', color='black', alpha=0.5, label='Teoría: p/4')
ax1.axhline(y=11, color='#8e44ad', linestyle=':', linewidth=2, label='Umbral seguridad 11%')
ax1.set_xlabel('Probabilidad de intercepción de Eva (%)')
ax1.set_ylabel('QBER (%)')
ax1.set_title('QBER en función de la actividad de Eva')
ax1.legend()
ax1.set_ylim(0, 30)

# Tasa de clave segura: R = 1 - 2h(QBER) donde h es la entropía binaria
def binary_entropy(e):
    if e <= 0 or e >= 1:
        return 0.0
    return -e * np.log2(e) - (1-e) * np.log2(1-e)

key_rates_secure = [max(0, 1 - 2*binary_entropy(q)) for q in qbers]
ax2.plot(eve_probs * 100, key_rates_secure, 's-', color='#2ecc71', label='Tasa segura R = 1 - 2h(QBER)')
ax2.axvline(x=44, color='#8e44ad', linestyle=':', linewidth=2, label='Eva >44%: R=0')
ax2.set_xlabel('Probabilidad de intercepción de Eva (%)')
ax2.set_ylabel('Tasa de clave segura R')
ax2.set_title('Tasa de clave segura vs presencia de Eva')
ax2.legend()
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()
print('Para QBER > 11%, la clave generada no es segura (Eva tiene demasiada información)')

## 3. Amplificación de Privacidad (Simulada)

In [ ]:
def privacy_amplification(key: np.ndarray, compression_factor: float) -> np.ndarray:
    """
    Amplificación de privacidad simplificada usando XOR de pares de bits.
    En la práctica se usa hashing universal.
    """
    n_output = int(len(key) * compression_factor)
    output = []
    for i in range(n_output):
        # XOR de dos bits de la clave cruda
        i1, i2 = 2*i % len(key), (2*i+1) % len(key)
        output.append(key[i1] ^ key[i2])
    return np.array(output)

# Protocolo completo con baja presencia de Eva
print('Protocolo BB84 completo con Eva al 10%:')
ak, bk, qber, sift_rate = bb84_protocol(100000, eve_intercept_prob=0.1, verbose=True)

# Verificación de un subconjunto para estimar QBER
check_fraction = 0.1
n_check = int(len(ak) * check_fraction)
ak_check, bk_check = ak[:n_check], bk[:n_check]
qber_estimated = float(np.mean(ak_check != bk_check))
ak_remaining, bk_remaining = ak[n_check:], bk[n_check:]

print(f'\nFase de verificación ({check_fraction:.0%} de la clave):')
print(f'  QBER estimado: {qber_estimated:.4f}')

if qber_estimated > 0.11:
    print('  ALERTA: QBER > 11%. Clave descartada por posible espionaje.')
else:
    print(f'  OK: QBER < 11%. Continuando...')
    
    # Amplificación de privacidad
    compression = max(0, 1 - 2*binary_entropy(qber_estimated))
    ak_final = privacy_amplification(ak_remaining, compression)
    bk_final = privacy_amplification(bk_remaining, compression)
    
    # Verificar que Alice y Bob tienen la misma clave
    key_error_rate = float(np.mean(ak_final != bk_final))
    
    print(f'\nClave final:')
    print(f'  Bits en clave cruda (después de verificación): {len(ak_remaining)}')
    print(f'  Factor de compresión: {compression:.4f}')
    print(f'  Bits en clave final: {len(ak_final)}')
    print(f'  Error en clave final: {key_error_rate:.6f}')
    print(f'  Primeros 20 bits de Alice: {ak_final[:20]}')
    print(f'  Primeros 20 bits de Bob:   {bk_final[:20]}')

## 4. Resumen del Protocolo BB84

In [ ]:
print('=== RESUMEN DEL PROTOCOLO BB84 ===')
print()
print('Fase 1 (CUÁNTICA): Alice envía fotones codificados en bases aleatorias')
print('Fase 2 (CUÁNTICA): Bob mide en bases aleatorias')
print('Fase 3 (CLÁSICA):  Alice y Bob comparan bases públicamente → sifting (~50% se descartan)')
print('Fase 4 (CLÁSICA):  Verificación de QBER en un subconjunto (~10% se descarta)')
print('Fase 5 (CLÁSICA):  Amplificación de privacidad → clave final segura')
print()
print('Eficiencia total aproximada:')
print('  100 fotones transmitidos')
print('  → ~50 bits tras sifting')
print('  → ~45 bits tras verificación (10% descartado)')
print('  → ~35-40 bits de clave final (tras amplificación de privacidad con QBER~5%)')
print()
print('Seguridad: garantizada por las leyes de la mecánica cuántica (teorema de no-clonación)')
print('           y la detección de errores por el QBER')